# Denoising autoencoder vs filter klasik
        **vision › denoising** · notebook `denoising_autoencoder`

        **Kapan dipakai:** Gambar ber-noise dan tersedia pasangan bersih untuk pelatihan; bandingkan dengan Gaussian/median filter.

        **Jebakan umum:** Noise saat latih harus mirip noise saat uji. Autoencoder kecil bisa mengaburkan detail; ukur dengan PSNR/SSIM dan lihat gambarnya.

> ⚠️ Notebook ini berat (butuh torch / bobot / GPU). Lihat sel pertama.

**Data demo:** CIFAR-10 subset (4000 latih / 1000 uji, 10 kelas, 32x32).

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
DATA_ROOT = ROOT / "data_latihan/vision/classification/cifar10_small"
NORMAL = None      # kelas yang dianggap NORMAL (hanya ini yang boleh dilihat saat melatih)
SEED, EPOCHS = 42, 25
MODEL_KEY = "denoising_autoencoder"
OUT = out_dir("vision", "denoising", MODEL_KEY); set_seed(SEED)

## 2. Data

In [ ]:
import torch, torch.nn as nn
from superdb.vision import load_images, device, show_grid
tr = pd.read_csv(DATA_ROOT / "train.csv"); te = pd.read_csv(DATA_ROOT / "test.csv").merge(pd.read_csv(DATA_ROOT / "test_labels.csv"), on="id")
X, Xte = load_images(tr["path"], DATA_ROOT), load_images(te["path"], DATA_ROOT); y, yte = tr["label"].to_numpy(), te["label"].to_numpy()
print(X.shape, Xte.shape, "| device:", device())

## 3. Buat versi ber-noise dan latih denoising autoencoder
Noise Gaussian σ dipakai saat latih dan uji. Pembanding: Gaussian filter dan median filter (tanpa belajar).

In [ ]:
SIGMA = 0.1
to01 = lambda a: torch.tensor(a).permute(0, 3, 1, 2).float() / 255
g = torch.Generator().manual_seed(SEED); noisy = lambda t: (t + SIGMA * torch.randn(t.shape, generator=g)).clamp(0, 1)
Ctr, Cte = to01(X), to01(Xte[:500]); Ntr, Nte = noisy(Ctr), noisy(Cte)
dev = device()
net = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(), nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
                    nn.ConvTranspose2d(64, 32, 2, stride=2), nn.ReLU(), nn.Conv2d(32, 3, 3, padding=1), nn.Sigmoid()).to(dev)
opt = torch.optim.Adam(net.parameters(), lr=2e-3)
for ep in range(EPOCHS):
    perm = torch.randperm(len(Ctr))
    for i in range(0, len(perm), 128):
        b = perm[i:i + 128]; loss = nn.functional.mse_loss(net(Ntr[b].to(dev)), Ctr[b].to(dev)); opt.zero_grad(); loss.backward(); opt.step()
net.eval()
with torch.no_grad(): den = net(Nte.to(dev)).cpu()
print("selesai latih; loss akhir:", round(loss.item(), 5))

## 4. Bandingkan dengan filter klasik (PSNR dan SSIM; lebih tinggi lebih baik)

In [ ]:
from scipy.ndimage import gaussian_filter, median_filter
from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim
C = Cte.permute(0, 2, 3, 1).numpy(); N = Nte.permute(0, 2, 3, 1).numpy(); Dn = den.permute(0, 2, 3, 1).numpy()
gf = np.stack([gaussian_filter(n, sigma=(0.8, 0.8, 0)) for n in N]); mf = np.stack([median_filter(n, size=(3, 3, 1)) for n in N])
score = lambda P: (np.mean([psnr(c, p, data_range=1) for c, p in zip(C, P)]), np.mean([ssim(c, p, channel_axis=2, data_range=1) for c, p in zip(C, P)]))
res = pd.DataFrame({"ber-noise (tanpa proses)": score(N), "Gaussian filter": score(gf), "Median filter": score(mf), "Denoising autoencoder": score(Dn)}, index=["PSNR", "SSIM"]).T
display(res.round(3)); metrics = res.round(4).to_dict("index")

In [ ]:
fig, ax = plt.subplots(5, 6, figsize=(9, 8))
for r, (nm, A) in enumerate((("bersih", C), ("ber-noise", N), ("Gaussian", gf), ("median", mf), ("autoencoder", Dn))):
    for c in range(6): ax[r, c].imshow(np.clip(A[c], 0, 1)); ax[r, c].axis("off")
    ax[r, 0].set_title(nm, fontsize=8, loc="left")
plt.tight_layout(); plt.savefig(OUT / "denoise.png", dpi=130); plt.show()
torch.save(net.state_dict(), OUT / "model.pt"); save_json({"metrics": metrics, "sigma": SIGMA}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Anomali visual nyata (cacat produk): fitur patch-level (PatchCore) jauh lebih peka daripada fitur seluruh gambar.
- Evaluasi anomali selalu dengan PR-AUC; ROC-AUC bisa menyesatkan bila anomali langka.